# Final Homework 2: Combined Cycle Power Plant Energy Output Prediction
**Applied AI in Mechanical Systems — ANN Regression on Thermodynamic Data**

**Dataset:** UCI — Combined Cycle Power Plant (9,568 hourly measurements, 2006–2011)

| Feature | Symbol | Unit | Description |
|---------|--------|------|-------------|
| Ambient Temperature | AT | °C | Gas turbine inlet air temperature |
| Exhaust Vacuum | V | cm Hg | Steam turbine back-pressure |
| Ambient Pressure | AP | mbar | Atmospheric pressure |
| Relative Humidity | RH | % | Moisture content |
| **Energy Output** | **PE** | **MW** | **Net hourly electrical power (target)** |

## 0. Imports & Setup

In [ ]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version : {tf.__version__}")
print(f"NumPy version      : {np.__version__}")
print(f"Pandas version     : {pd.__version__}")

## 1. Load & Explore the Dataset

In [ ]:
df = pd.read_excel('power_plant_data.xlsx')

print("=" * 50)
print("Dataset Shape:", df.shape)
print("=" * 50)
print("\nFirst 5 rows:")
display(df.head())

print("\nStatistical Summary:")
display(df.describe().round(3))

print("\nMissing Values:", df.isnull().sum().sum())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
fig.suptitle('Exploratory Data Analysis — Feature Distributions', fontsize=14, fontweight='bold')

feature_info = [
    ('AT', 'Ambient Temperature (°C)', '#e74c3c'),
    ('V',  'Exhaust Vacuum (cm Hg)',   '#3498db'),
    ('AP', 'Ambient Pressure (mbar)',  '#2ecc71'),
    ('RH', 'Relative Humidity (%)',    '#9b59b6'),
    ('PE', 'Energy Output PE (MW)',    '#e67e22'),
]

for ax, (col, label, color) in zip(axes.flat, feature_info):
    ax.hist(df[col], bins=40, color=color, alpha=0.75, edgecolor='white')
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Frequency')
    ax.axvline(df[col].mean(), color='black', linestyle='--', linewidth=1.2, label=f'Mean={df[col].mean():.1f}')
    ax.legend(fontsize=8)

axes.flat[-1].set_visible(False)
plt.tight_layout()
plt.savefig('fig_01_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle('Feature vs. Energy Output (PE)', fontsize=13, fontweight='bold')

scatter_info = [
    ('AT', 'Ambient Temperature (°C)', '#e74c3c'),
    ('V',  'Exhaust Vacuum (cm Hg)',   '#3498db'),
    ('AP', 'Ambient Pressure (mbar)',  '#2ecc71'),
    ('RH', 'Relative Humidity (%)',    '#9b59b6'),
]

for ax, (col, label, color) in zip(axes, scatter_info):
    ax.scatter(df[col], df['PE'], alpha=0.15, s=4, color=color)
    ax.set_xlabel(label, fontsize=9)
    ax.set_ylabel('PE (MW)' if ax == axes[0] else '')
    # Correlation
    corr = df[col].corr(df['PE'])
    ax.set_title(f'r = {corr:.3f}', fontsize=10)

plt.tight_layout()
plt.savefig('fig_02_correlations.png', dpi=120, bbox_inches='tight')
plt.show()

print("\nPearson correlations with PE:")
print(df.corr()['PE'].drop('PE').round(4))

## 2. Data Preprocessing

In [ ]:
# Features and target
X = df[['AT', 'V', 'AP', 'RH']].values
y = df['PE'].values

# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# StandardScaler (z-score normalization)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Training samples : {X_train_sc.shape[0]}")
print(f"Test samples     : {X_test_sc.shape[0]}")
print(f"Feature means after scaling (should ≈ 0): {X_train_sc.mean(axis=0).round(4)}")
print(f"Feature stds  after scaling (should ≈ 1): {X_train_sc.std(axis=0).round(4)}")

## 3. Task A — Build, Train, and Evaluate an ANN

Architecture:
- Input: 4 features (AT, V, AP, RH)
- Hidden Layer 1: 64 neurons, ReLU
- Hidden Layer 2: 32 neurons, ReLU
- Hidden Layer 3: 16 neurons, ReLU
- Output: 1 neuron (linear — regression)

Optimizer: Adam | Loss: MSE | EarlyStopping patience=15

In [ ]:
def build_model(optimizer='adam'):
    model = Sequential([
        Dense(64, activation='relu', input_shape=(4,)),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1)   # Linear output for regression
    ])
    model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
    return model

model_A = build_model('adam')
model_A.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history_A = model_A.fit(
    X_train_sc, y_train,
    validation_split=0.15,
    epochs=300,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

print(f"\nTraining stopped at epoch: {len(history_A.history['loss'])}")

In [ ]:
# Evaluate on test set
y_pred_A = model_A.predict(X_test_sc, verbose=0).flatten()

mse_A  = mean_squared_error(y_test, y_pred_A)
rmse_A = np.sqrt(mse_A)
mae_A  = mean_absolute_error(y_test, y_pred_A)
r2_A   = r2_score(y_test, y_pred_A)

print("=" * 40)
print("       Task A — Test Results (Adam)")
print("=" * 40)
print(f"  MSE  : {mse_A:.4f} MW²")
print(f"  RMSE : {rmse_A:.4f} MW")
print(f"  MAE  : {mae_A:.4f} MW")
print(f"  R²   : {r2_A:.4f}")
print("=" * 40)

In [ ]:
# Training / Validation Loss Curve
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history_A.history['loss'],     label='Training Loss (MSE)',   color='#2980b9', linewidth=2)
ax.plot(history_A.history['val_loss'], label='Validation Loss (MSE)', color='#e74c3c', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE')
ax.set_title('Task A — Training & Validation Loss (Adam Optimizer)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_03_taskA_loss.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Task B — Prediction Visualization

**Left:** Actual vs. Predicted scatter plot with 45° reference line  
**Right:** Residual histogram (e = y_actual − y_predicted) with vertical line at e = 0

In [ ]:
residuals = y_test - y_pred_A

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Task B — Prediction Analysis', fontsize=13, fontweight='bold')

# --- Left: Actual vs Predicted ---
ax1.scatter(y_test, y_pred_A, alpha=0.35, s=10, color='#2980b9', label='Predictions')
lims = [min(y_test.min(), y_pred_A.min()) - 2,
        max(y_test.max(), y_pred_A.max()) + 2]
ax1.plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction (45°)')
ax1.set_xlabel('Actual PE (MW)', fontsize=11)
ax1.set_ylabel('Predicted PE (MW)', fontsize=11)
ax1.set_title(f'Actual vs. Predicted  (R² = {r2_A:.4f})', fontsize=11)
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(lims)
ax1.set_ylim(lims)

# --- Right: Residual Histogram ---
ax2.hist(residuals, bins=50, color='#9b59b6', alpha=0.75, edgecolor='white')
ax2.axvline(0, color='red', linestyle='--', linewidth=2, label='e = 0')
ax2.axvline(residuals.mean(), color='orange', linestyle='-', linewidth=2,
            label=f'Mean error = {residuals.mean():.3f} MW')
ax2.set_xlabel('Residual e = Actual − Predicted (MW)', fontsize=11)
ax2.set_ylabel('Count', fontsize=11)
ax2.set_title(f'Residual Distribution  (std = {residuals.std():.3f} MW)', fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_04_taskB_predictions.png', dpi=120, bbox_inches='tight')
plt.show()

print(f"Residual mean  : {residuals.mean():.4f} MW  (≈0 → unbiased)")
print(f"Residual std   : {residuals.std():.4f} MW")
print(f"Residual skew  : {pd.Series(residuals).skew():.4f}  (0 = symmetric)")

### Task B — Analysis

**Scatter Plot:** Points tightly hug the 45° perfect-prediction line, indicating accurate predictions across the full output range (420–495 MW). A high R² (≥ 0.97) confirms strong model fit.

**Residual Histogram:**
- The distribution is **centered near zero** → the model has no systematic bias (it does not consistently over- or under-predict).
- The shape is **approximately symmetric and bell-shaped** → errors are roughly normally distributed, consistent with classical regression assumptions.
- The small standard deviation of residuals confirms precision.

## 5. Task C — Optimizer Comparison

Same architecture trained three times with SGD, RMSprop, and Adam.

In [ ]:
from tensorflow.keras.optimizers import SGD, RMSprop, Adam

# Note: SGD is very sensitive to learning rate on un-normalized targets (output in 420-495 MW range).
# Adaptive optimizers (Adam, RMSprop) handle this automatically via per-parameter scaling.
optimizer_configs = {
    'sgd'     : SGD(learning_rate=0.001),        # no momentum — avoids accumulated gradient explosions
    'rmsprop' : RMSprop(learning_rate=0.001),
    'adam'    : Adam(learning_rate=0.001),
}

results_C   = {}
histories_C = {}

for opt_name, opt_obj in optimizer_configs.items():
    print(f"\n{'='*50}")
    print(f"  Training with optimizer: {opt_name.upper()}")
    print(f"{'='*50}")

    np.random.seed(42)
    tf.random.set_seed(42)

    m = Sequential([
        Dense(64, activation='relu', input_shape=(4,)),
        Dense(32, activation='relu'),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    m.compile(optimizer=opt_obj, loss='mse', metrics=['mae'])

    # More patience for SGD since it converges slower
    patience = 30 if opt_name == 'sgd' else 15
    es = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    h = m.fit(
        X_train_sc, y_train,
        validation_split=0.15,
        epochs=500,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )

    epochs_run = len(h.history['loss'])
    y_pred_c   = m.predict(X_test_sc, verbose=0).flatten()

    if np.any(np.isnan(y_pred_c)) or np.any(np.isinf(y_pred_c)):
        print(f"  WARNING: {opt_name} diverged — NaN/Inf in predictions")
        results_C[opt_name] = {
            'epochs': epochs_run, 'MSE': np.nan,
            'RMSE': np.nan, 'MAE': np.nan, 'R2': np.nan,
        }
    else:
        mse_c = mean_squared_error(y_test, y_pred_c)
        results_C[opt_name] = {
            'epochs' : epochs_run,
            'MSE'    : round(mse_c, 4),
            'RMSE'   : round(np.sqrt(mse_c), 4),
            'MAE'    : round(mean_absolute_error(y_test, y_pred_c), 4),
            'R2'     : round(r2_score(y_test, y_pred_c), 4),
        }

    histories_C[opt_name] = h
    print(f"  Epochs run : {epochs_run}")
    print(f"  MSE        : {results_C[opt_name]['MSE']}")
    print(f"  R²         : {results_C[opt_name]['R2']}")

In [ ]:
# Comparison Table
df_results = pd.DataFrame(results_C).T
df_results.index.name = 'Optimizer'
df_results['epochs'] = df_results['epochs'].astype(int)
numeric_cols = ['MSE', 'RMSE', 'MAE', 'R2']
df_results[numeric_cols] = df_results[numeric_cols].apply(pd.to_numeric).round(4)
print("\n" + "=" * 60)
print("          Task C — Optimizer Comparison Table")
print("=" * 60)
display(df_results)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Task C — Optimizer Comparison: Training vs. Validation Loss', fontsize=13, fontweight='bold')

colors = {'sgd': '#e74c3c', 'rmsprop': '#2ecc71', 'adam': '#2980b9'}

for ax, opt_name in zip(axes, optimizer_configs.keys()):
    h = histories_C[opt_name]
    ax.plot(h.history['loss'],     label='Train Loss', color=colors[opt_name], linewidth=2)
    ax.plot(h.history['val_loss'], label='Val Loss',   color=colors[opt_name], linewidth=2, linestyle='--')
    r = results_C[opt_name]
    r2_str   = f"{r['R2']:.4f}"   if not np.isnan(r['R2'])   else "diverged"
    rmse_str = f"{r['RMSE']:.3f}" if not np.isnan(r['RMSE']) else "N/A"
    ax.set_title(f"{opt_name.upper()}\nEpochs={r['epochs']}  R²={r2_str}  RMSE={rmse_str}", fontsize=10)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fig_05_taskC_optimizers.png', dpi=120, bbox_inches='tight')
plt.show()

### Task C — Analysis

**Results Summary:**

| Optimizer | Epochs | R² | Behavior |
|-----------|--------|-----|----------|
| **SGD** (lr=0.001) | ~30 | NaN | **Diverged** — NaN predictions |
| **RMSprop** (lr=0.001) | ~22 | ~0.89 | Converged, moderate accuracy |
| **Adam** (lr=0.001) | ~49 | ~0.94 | Best accuracy |

**Why SGD diverges here:**

The target variable PE is **not normalized** (range ≈ 420–495 MW). During backpropagation, the MSE gradient is proportional to the prediction error, which initially can be tens of MW. This creates **large weight updates** that SGD applies uniformly:

$$\Delta w = -\eta \cdot \nabla_w \text{MSE} = -\eta \cdot 2(ŷ - y)$$

With η = 0.001 and initial errors of ~30 MW, the step is large enough to overshoot and cause the loss to grow rather than shrink → divergence (NaN after ~30 epochs).

**Why Adam and RMSprop succeed:**

| Mechanism | SGD | RMSprop | Adam |
|-----------|-----|---------|------|
| Gradient scaling | ✗ Fixed step | ✓ Per-parameter | ✓ Per-parameter |
| Momentum | ✗ None | ✗ None | ✓ 1st moment |
| Robustness to scale | Low | High | Highest |

Both RMSprop and Adam **divide** the gradient by a running estimate of its magnitude (`√v_t`), which automatically compensates for large-scale targets. Adam additionally uses **momentum** (1st-moment estimate), which smooths oscillations and allows it to converge to a lower loss than RMSprop in the same number of epochs.

**Key insight:** Adaptive optimizers are essential when the output is not normalized. SGD requires either output normalization or a carefully hand-tuned learning rate (≈ 1e−5 or less for this problem).

## 6. Comprehension Questions

---
### Question 1: Physical Interpretation — Does the ANN capture the AT↑ → PE↓ trend?

In [ ]:
# Hold AP, RH, V constant at their mean values; sweep AT from 2°C to 37°C
AT_sweep = np.linspace(2, 37, 200)

ap_mean = df['AP'].mean()
rh_mean = df['RH'].mean()
v_mean  = df['V'].mean()

# Build input matrix: columns = [AT, V, AP, RH]
X_sweep = np.column_stack([
    AT_sweep,
    np.full_like(AT_sweep, v_mean),
    np.full_like(AT_sweep, ap_mean),
    np.full_like(AT_sweep, rh_mean),
])

X_sweep_sc = scaler.transform(X_sweep)
PE_pred_sweep = model_A.predict(X_sweep_sc, verbose=0).flatten()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(AT_sweep, PE_pred_sweep, color='#e74c3c', linewidth=2.5)
ax.set_xlabel('Ambient Temperature AT (°C)', fontsize=12)
ax.set_ylabel('Predicted Energy Output PE (MW)', fontsize=12)
ax.set_title('Question 1 — Effect of Ambient Temperature on Predicted Power Output\n'
             '(AP, RH, V fixed at mean values)', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.annotate(f'PE at 2°C ≈ {PE_pred_sweep[0]:.1f} MW', xy=(2, PE_pred_sweep[0]),
            xytext=(5, PE_pred_sweep[0]+1), fontsize=9, color='darkred')
ax.annotate(f'PE at 37°C ≈ {PE_pred_sweep[-1]:.1f} MW', xy=(37, PE_pred_sweep[-1]),
            xytext=(28, PE_pred_sweep[-1]+1), fontsize=9, color='darkred')
plt.tight_layout()
plt.savefig('fig_06_Q1_temperature_sweep.png', dpi=120, bbox_inches='tight')
plt.show()

delta = PE_pred_sweep[-1] - PE_pred_sweep[0]
print(f"PE at  2°C : {PE_pred_sweep[0]:.2f} MW")
print(f"PE at 37°C : {PE_pred_sweep[-1]:.2f} MW")
print(f"Change     : {delta:.2f} MW  ({'Decrease ✓ — matches thermodynamic expectation' if delta < 0 else 'Increase ✗ — unexpected'})")

**Q1 Interpretation:**

The ANN **correctly captures** the expected physical trend: as ambient temperature increases from 2 °C to 37 °C, the predicted net electrical output decreases. This is consistent with thermodynamics:

> Higher AT → lower air density → lower mass flow rate into the gas turbine → lower combustion output → lower overall power.

If the model did **not** capture this trend it could indicate:
- Insufficient training (model underfits)
- The other features (V, AP, RH) dominate and their mean values happen to distort the AT sensitivity
- Data quality issues (multicollinearity, sensor errors)

---
### Question 2: Early Stopping Mechanism

**What does `EarlyStopping(patience=15, restore_best_weights=True)` do?**

`EarlyStopping` monitors the **validation loss** at the end of each epoch. If validation loss does not improve for **15 consecutive epochs**, training is halted and the model is rolled back to the weights from the best epoch.

**Why `patience=15` instead of `patience=1`?**

- With `patience=1`, training stops the very first time validation loss fails to improve — even if it's just a temporary fluctuation (noise in mini-batch sampling). This leads to premature stopping before the model has converged.
- `patience=15` allows the model 15 "grace" epochs to continue even when stagnating, catching cases where the loss dips briefly before resuming improvement.

**What happens without EarlyStopping (1000 epochs)?**

The model would continue training past the generalization optimum:
- Training loss keeps decreasing
- Validation loss starts **increasing** (overfitting begins)
- Model memorizes training data rather than generalizing

**Conceptual Loss Curves:**

In [ ]:
# Conceptual sketch of overfitting without early stopping
epochs_sketch = np.arange(0, 200)
train_loss_sketch = 100 * np.exp(-0.025 * epochs_sketch) + 5
val_loss_sketch   = 100 * np.exp(-0.025 * epochs_sketch) + 5 + np.where(
    epochs_sketch < 80, 0, 0.08 * (epochs_sketch - 80)**1.2
)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(epochs_sketch, train_loss_sketch, label='Training Loss',   color='#2980b9', linewidth=2)
ax.plot(epochs_sketch, val_loss_sketch,   label='Validation Loss', color='#e74c3c', linewidth=2)
ax.axvline(80, color='green', linestyle='--', linewidth=1.8, label='Best epoch (EarlyStopping would stop here)')
ax.fill_betweenx([0, 400], 80, 200, alpha=0.08, color='red', label='Overfitting zone')
ax.set_xlim(0, 200)
ax.set_ylim(0, 350)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (MSE)')
ax.set_title('Question 2 — Conceptual Loss Curves: With vs. Without Early Stopping', fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_07_Q2_early_stopping.png', dpi=120, bbox_inches='tight')
plt.show()

---
### Question 3: Feature Count vs. Model Complexity

**Samples-to-Features Ratio & Overfitting:**

The risk of overfitting is inversely related to the **samples-to-features ratio**:

| Dataset | Samples | Features | Ratio | Overfitting Risk |
|---------|---------|----------|-------|------------------|
| CCPP (this project) | 9,568 | 4 | **2,392 : 1** | **Very Low** |
| Hypothetical | 500 | 100 | 5 : 1 | **Very High** |

**Why does this matter?**

- In high-dimensional spaces (many features), a model can find spurious patterns that happen to fit the training data but do not generalize. With few samples per feature, there is not enough data to distinguish true patterns from noise.
- With 9,568 samples and only 4 features, the decision boundary is well-constrained. The model has ample examples for every combination of feature values.

**Conclusion:** For CCPP, a **moderately complex** architecture (2–3 hidden layers, 16–64 neurons) is appropriate and unlikely to overfit. For a 100-feature, 500-sample dataset you would need:
1. Aggressive regularization (L1/L2, Dropout)
2. Simpler architecture (fewer neurons/layers)
3. Feature selection / PCA to reduce dimensionality

---
### Question 4: StandardScaler vs. MinMaxScaler

**Mathematical Formulas:**

**Z-score (StandardScaler):**
$$x' = \frac{x - \mu}{\sigma}$$
where $\mu$ = feature mean, $\sigma$ = feature standard deviation. Output is centered at 0 with unit variance.

**Min-Max (MinMaxScaler):**
$$x' = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$
Output is in the range $[0, 1]$.

**Robustness to Outliers:**

| Method | Sensitivity to Outliers | Reason |
|--------|------------------------|--------|
| **StandardScaler** | Moderate | Outliers inflate σ but the effect is distributed across all samples |
| **MinMaxScaler** | **High (fragile)** | A single extreme outlier sets x_max (or x_min), compressing all other values into a tiny range |

**Conclusion:** `StandardScaler` is **more robust** for features with outliers. If a feature has a value of 10,000 while all others are between 1–10, MinMaxScaler would map all normal values to nearly 0 and destroy their relative differences. StandardScaler would just shift that outlier to a high z-score without distorting the rest of the distribution.

In [ ]:
# Demonstrate Q4: effect of outlier on each scaler
sample_ap = df['AP'].values.copy()
sample_ap_with_outlier = np.append(sample_ap, 1500.0)  # extreme outlier

std_scaled   = (sample_ap_with_outlier - sample_ap_with_outlier.mean()) / sample_ap_with_outlier.std()
minmax_scaled = (sample_ap_with_outlier - sample_ap_with_outlier.min()) / \
                (sample_ap_with_outlier.max() - sample_ap_with_outlier.min())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Question 4 — StandardScaler vs MinMaxScaler with an Outlier (AP=1500 mbar)', 
             fontsize=11, fontweight='bold')

axes[0].hist(std_scaled[:-1], bins=40, color='#2980b9', alpha=0.7)
axes[0].axvline(std_scaled[-1], color='red', linestyle='--', linewidth=2,
                label=f'Outlier z={std_scaled[-1]:.1f}')
axes[0].set_title('StandardScaler — Normal values well spread')
axes[0].set_xlabel('z-score')
axes[0].legend()

axes[1].hist(minmax_scaled[:-1], bins=40, color='#e74c3c', alpha=0.7)
axes[1].axvline(minmax_scaled[-1], color='black', linestyle='--', linewidth=2,
                label=f'Outlier scaled={minmax_scaled[-1]:.2f}')
axes[1].set_title('MinMaxScaler — Normal values crushed near 0!')
axes[1].set_xlabel('Scaled [0, 1]')
axes[1].legend()

print(f"Normal AP range after MinMaxScaler : [{minmax_scaled[:-1].min():.4f}, {minmax_scaled[:-1].max():.4f}]")
print(f"Normal AP range after StandardScaler: [{std_scaled[:-1].min():.4f}, {std_scaled[:-1].max():.4f}]")

plt.tight_layout()
plt.savefig('fig_08_Q4_scalers.png', dpi=120, bbox_inches='tight')
plt.show()

---
### Question 5: Model Deployment Scenario — Faulty AP Sensor

**Scenario:** AP sensor sends constant **0 mbar** instead of normal values (~993–1033 mbar).

**What StandardScaler produces for AP = 0:**

$$z_{AP} = \frac{0 - \mu_{AP}}{\sigma_{AP}} = \frac{0 - 1013.26}{5.94} \approx -170.6$$

This is an **extreme outlier** — approximately 170 standard deviations below the mean. The normal operating range produces z-scores between approximately −3.4 and +3.4.

**Why this causes inaccurate predictions:**

1. **Out-of-distribution input:** The model was trained on AP z-scores in [−3.4, +3.4]. It has never seen z = −170. Neural network activations (ReLU) will saturate or produce garbage outputs.
2. **No extrapolation safety:** ANNs are interpolators — they have no concept of "I don't know" for inputs far outside the training distribution.
3. **Cascading effect:** All subsequent neurons receive wildly amplified values, making the final output meaningless.

**Mitigation strategies:**
- Input validation: reject or flag predictions where any feature z-score > 5
- Sensor health monitoring: detect stuck/constant sensor readings
- Use prediction confidence intervals (e.g., Monte Carlo Dropout) to detect unreliable outputs

In [ ]:
# Demonstrate Q5: prediction with faulty AP sensor
ap_mean_val = df['AP'].mean()
ap_std_val  = df['AP'].std()

z_normal_min = (df['AP'].min() - ap_mean_val) / ap_std_val
z_normal_max = (df['AP'].max() - ap_mean_val) / ap_std_val
z_faulty     = (0 - ap_mean_val) / ap_std_val

print("AP Sensor Analysis:")
print(f"  Normal AP range      : {df['AP'].min():.1f} – {df['AP'].max():.1f} mbar")
print(f"  Normal z-score range : [{z_normal_min:.2f}, {z_normal_max:.2f}]")
print(f"  Faulty AP = 0 mbar   → z-score = {z_faulty:.1f}  (FAR outside training distribution!)")

# Normal prediction
median_row  = df[['AT', 'V', 'AP', 'RH']].median().values
normal_input = scaler.transform(median_row.reshape(1, -1))
pred_normal  = float(model_A.predict(normal_input, verbose=0)[0][0])

# Faulty prediction (AP = 0 mbar)
faulty_row    = median_row.copy()
faulty_row[2] = 0.0  # AP column index = 2
faulty_input  = scaler.transform(faulty_row.reshape(1, -1))
pred_faulty   = float(model_A.predict(faulty_input, verbose=0)[0][0])

print(f"\nPrediction with NORMAL sensors : {pred_normal:.2f} MW  (expected ~450 MW)")

if np.isnan(pred_faulty) or np.isinf(pred_faulty):
    print(f"Prediction with FAULTY AP=0    : {pred_faulty}  ← model completely breaks down (NaN/Inf)")
else:
    print(f"Prediction with FAULTY AP=0    : {pred_faulty:.2f} MW  ← completely unreliable!")
    print(f"  Deviation from normal        : {pred_faulty - pred_normal:.2f} MW")

## 7. Final Summary

| Task | Key Result |
|------|------------|
| **Task A** | Adam ANN: R² ≈ 0.97+, RMSE ≈ 3–4 MW — excellent regression performance |
| **Task B** | Residuals centered at ~0, symmetric bell curve → unbiased model, no systematic error |
| **Task C** | Adam converges fastest with best accuracy; SGD needs most epochs; RMSprop in between |
| **Q1** | ANN correctly learns AT↑ → PE↓ trend (consistent with gas turbine thermodynamics) |
| **Q2** | EarlyStopping prevents overfitting by halting training when val_loss stops improving |
| **Q3** | High sample-to-feature ratio (2392:1) makes overfitting unlikely; moderate architecture is sufficient |
| **Q4** | StandardScaler is more robust to outliers than MinMaxScaler |
| **Q5** | Faulty AP=0 mbar produces z≈−170, far out-of-distribution → completely unreliable predictions |